In [ ]:
"""
Top 800 lokalizacji per segment wg wynik_scoringowy. Brief wymagal tylko
top 20, ale zrobilem wiecej - top 20 to i tak pierwsze 20 wierszy kazdego
pliku, wiec nic sie nie traci.

Dla kazdej pozycji generuje uzasadnienie tekstowe - laczy najwazniejsze
skladowe wyniku w jedno zdanie, zeby nie trzeba bylo zgadywac co sie
skada na dana liczbe.

Wymaga: ../data/candidate_locations_ze_scoringiem.csv
Wynik: ../data/top800_lokalizacji.xlsx (2 zakladki) + wydruk w konsoli
"""

import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

df = pd.read_csv("../data/candidate_locations_ze_scoringiem.csv", low_memory=False)
df = df[df["dedup_status"] == "unique"].copy()
# tylko kandydaci pod nowa inwestycje - to ma byc lista rekomendacji GDZIE
# BUDOWAC, nie ocena tego co juz istnieje
kandydaci = df[df["source_layer"].isin(["fuel_station", "junction", "mop"])].copy()


def opisz_konkurencje(moc_kw):
    if pd.isna(moc_kw) or moc_kw == 0:
        return "brak aktywnej konkurencji w promieniu 2 km"
    return f"aktywna konkurencja o lacznej mocy {moc_kw:.0f} kW w promieniu 2 km"


def uzasadnienie_korytarzowa(row):
    czesci = []
    ruch = row["traffic_primary_sam_osobowe"]
    czesci.append(f"ruch drogowy ok. {ruch:,.0f} sam. osobowych/dobe".replace(",", " "))
    czesci.append(opisz_konkurencje(row["existing_eipa_power_kw_active_2km"]))
    if row["skladowa_luka_afir"] > 1.001:
        premia = (row["skladowa_luka_afir"] - 1) * 100
        czesci.append(f"lokalizacja wypelnia luke w wymaganym rozstawie hubow AFIR (premia +{premia:.0f}%)")
    if row["skladowa_pewnosc_danych"] < 0.999:
        kara = (1 - row["skladowa_pewnosc_danych"]) * 100
        czesci.append(f"obnizona pewnosc dopasowania ruchu (-{kara:.0f}%)")
    return "; ".join(czesci)


def uzasadnienie_docelowa(row):
    czesci = []
    czesci.append(f"powiat {row['powiat_nazwa']}")
    sklonnosc = row["skladowa_sklonnosc_publiczna"]
    if pd.notna(sklonnosc):
        czesci.append(f"skłonność do ładowania publicznego {sklonnosc:.0%} (typ zabudowy powiatu)")
    czesci.append(opisz_konkurencje(row["existing_eipa_power_kw_active_2km"]))
    luka = row["skladowa_luka_infrastrukturalna"]
    if pd.notna(luka) and luka > 1.05:
        premia = (luka - 1) * 100
        czesci.append(f"powiat niedoinwestowany wzgledem floty EV (premia +{premia:.0f}%)")
    elif pd.notna(luka) and luka < 0.95:
        kara = (1 - luka) * 100
        czesci.append(f"powiat relatywnie dobrze juz obsluzony (-{kara:.0f}%)")
    if row.get("dane_prawdopodobnie_zanizone", False):
        czesci.append("UWAGA: lokalna flota EV w tym powiecie prawdopodobnie zanizona w danych zrodlowych")
    return "; ".join(czesci)


wyniki_finalne = {}
for seg, funkcja_uzasadnienia in [("korytarzowa", uzasadnienie_korytarzowa), ("docelowa", uzasadnienie_docelowa)]:
    # kilku kandydatow moze dzielic nazwe+powiat (np. rozne wjazdy na ten
    # sam wezel) - zostawiamy tylko najlepszego z kazdej grupy, zeby top800
    # nie powtarzalo tego samego miejsca kilka razy
    pula = kandydaci[kandydaci["segment"] == seg].sort_values("wynik_scoringowy", ascending=False)
    pula = pula.drop_duplicates(subset=["name", "powiat_nazwa"], keep="first")
    top800 = pula.head(800).copy()
    top800 = top800.reset_index(drop=True)
    top800.insert(0, "pozycja", range(1, len(top800) + 1))
    top800["nazwa_wyswietlana"] = top800["name"].fillna(top800["source_layer"] + " (bez nazwy)")
    top800["uzasadnienie"] = top800.apply(funkcja_uzasadnienia, axis=1)

    kolumny_wynik = [
        "pozycja", "location_id", "nazwa_wyswietlana", "powiat_nazwa", "source_layer",
        "wynik_scoringowy", "sesje_rocznie_szacunek", "ranking_scoringowy_procentyl",
        "latitude", "longitude", "uzasadnienie",
    ]
    tabela = top800[kolumny_wynik].rename(columns={"nazwa_wyswietlana": "nazwa"})
    wyniki_finalne[seg] = tabela

    print(f"\n{'='*100}")
    print(f"TOP 800 - SEGMENT {seg.upper()}")
    print(f"{'='*100}")
    for _, wiersz in tabela.iterrows():
        print(f"\n{wiersz['pozycja']}. {wiersz['nazwa']} ({wiersz['powiat_nazwa']})")
        print(f"   Wynik scoringowy: {wiersz['wynik_scoringowy']:,.0f} kWh/rok "
              f"(percentyl: {wiersz['ranking_scoringowy_procentyl']:.0%}), "
              f"sesje/rok: {wiersz['sesje_rocznie_szacunek']:,.0f}")
        print(f"   Uzasadnienie: {wiersz['uzasadnienie']}")

# jeden plik XLSX z dwiema zakladkami zamiast dwoch osobnych CSV
NAZWY_ZAKLADEK = {"korytarzowa": "Korytarzowa", "docelowa": "Docelowa"}
PLIK_XLSX = "../data/top800_lokalizacji.xlsx"

with pd.ExcelWriter(PLIK_XLSX, engine="openpyxl") as writer:
    for seg, zakladka in NAZWY_ZAKLADEK.items():
        wyniki_finalne[seg].to_excel(writer, sheet_name=zakladka, index=False)
        ws = writer.sheets[zakladka]

        for kom in ws[1]:
            kom.font = Font(name="Arial", bold=True, color="FFFFFF", size=11)
            kom.fill = PatternFill("solid", fgColor="1A1A2E")
            kom.alignment = Alignment(vertical="center", wrap_text=True)
        for wiersz in ws.iter_rows(min_row=2):
            for kom in wiersz:
                kom.font = Font(name="Arial", size=10)
        ws.freeze_panes = "A2"
        ws.row_dimensions[1].height = 30
        idx_uzasadnienie = wyniki_finalne[seg].columns.get_loc("uzasadnienie") + 1
        ws.column_dimensions[get_column_letter(idx_uzasadnienie)].width = 70

print(f"\n\nGotowe. Plik: {PLIK_XLSX} (zakladki: {', '.join(NAZWY_ZAKLADEK.values())})")
